In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Tweets.csv to Tweets.csv


In [ ]:
import pandas as pd
df = pd.read_csv("Tweets.csv")
df.shape

(14640, 15)

In [ ]:
df=df [["tweet_id", "airline", "airline_sentiment", "airline_sentiment_confidence", "negativereason", "text"]]
df.head()

,tweet_id,airline,airline_sentiment,airline_sentiment_confidence,negativereason,text
0,570306133677760513,Virgin America,neutral,1.0000,NaN,@VirginAmerica What @dhepburn said.
1,570301130888122368,Virgin America,positive,0.3486,NaN,@VirginAmerica plus you've added commercials t...
2,570301083672813571,Virgin America,neutral,0.6837,NaN,@VirginAmerica I didn't today... Must mean I n...
3,570301031407624196,Virgin America,negative,1.0000,Bad Flight,@VirginAmerica it's really aggressive to blast...
4,570300817074462722,Virgin America,negative,1.0000,Can't Tell,@VirginAmerica and it's a really big bad thing...


cell only has 6 from the dataset and leaves the others

In [ ]:
df["airline_sentiment"].value_counts()
df["airline"].value_counts()
pd.crosstab(df["airline"], df["airline_sentiment"], normalize="index").round(3)

airline_sentiment,negative,neutral,positive
airline,,,
American,0.710,0.168,0.122
Delta,0.430,0.325,0.245
Southwest,0.490,0.274,0.236
US Airways,0.777,0.131,0.092
United,0.689,0.182,0.129
Virgin America,0.359,0.339,0.302


remember panda doesnt accept any british spelling:(

In [ ]:
df["text"].duplicated().sum()
df["tweet_id"].duplicated().sum()

np.int64(155)

In [ ]:
df["airline_sentiment_confidence"].describe()
df.groupby("airline_sentiment")["airline_sentiment_confidence"].mean()

,airline_sentiment_confidence
airline_sentiment,
negative,0.933365
neutral,0.823303
positive,0.872039


In [ ]:
df["text"].str.len().describe()

,text
count,14640.000000
mean,103.822063
std,36.277339
min,12.000000
25%,77.000000
50%,114.000000
75%,136.000000
max,186.000000


In [ ]:
df.sample(10, random_state=42)[["airline_sentiment", "text"]]

,airline_sentiment,text
4794,positive,@SouthwestAir you're my early frontrunner for ...
10480,negative,@USAirways how is it that my flt to EWR was Ca...
8067,negative,@JetBlue what is going on with your BDL to DCA...
8880,neutral,@JetBlue do they have to depart from Washingto...
8292,negative,@JetBlue I can probably find some of them. Are...
927,negative,@united still waiting to hear back. My wallet ...
3165,negative,@united Yes my flight was rebooked. I'm just l...
7894,neutral,@JetBlue Thank you ! What about Paris ? Could ...
2818,negative,"@united not 100% sure, however my ticket inclu..."
9145,positive,@usairways great crew for flight 504 PHX to YV...


randomly chooses 10 rows of tweets

In [ ]:
delta=df[df["airline"] == "Delta"]
print(delta["text"].str.contains("JetBlue", case=False).mean())

0.9981998199819982


In [ ]:
print(df["airline_sentiment"].value_counts)
print(df["airline"].value_counts())
print(df["text"].duplicated().sum())
print(df["airline_sentiment_confidence"].describe())

<bound method IndexOpsMixin.value_counts of 0         neutral
1        positive
2         neutral
3        negative
4        negative
           ...   
14635    positive
14636    negative
14637     neutral
14638    negative
14639     neutral
Name: airline_sentiment, Length: 14640, dtype: object>
airline
United            3822
US Airways        2913
American          2759
Southwest         2420
Delta             2222
Virgin America     504
Name: count, dtype: int64
213
count    14640.000000
mean         0.900169
std          0.162830
min          0.335000
25%          0.692300
50%          1.000000
75%          1.000000
max          1.000000
Name: airline_sentiment_confidence, dtype: float64


double equals == is for comparison

In [ ]:
#correcting delta, jetblue tagging anomoly
df["airline_clean"]=df["airline"]
is_jb=(df["airline"]=="Delta")& df["text"].str.contains("JetBlue", case=False)
df.loc[is_jb, "airline_clean"]="JetBlue"
df=df[df["airline_clean"]!="Delta"]
print(df["airline_clean"].value_counts())


airline_clean
United            3780
US Airways        2882
American          2574
Southwest         2384
JetBlue           2161
Virgin America     491
Name: count, dtype: int64


In [ ]:
#dropping duplucate tweet ids & bot retweets
df=df.sort_values(["airline_sentiment_confidence", "tweet_id"],
                  ascending=[False,True],kind="stable")
df=df.drop_duplicates(subset="tweet_id", keep="first")
print("after tweet_id dedupe:", len(df))
df=df.drop_duplicates(subset="text", keep="first")
print("after text dedupe:", len(df))
df=df.reset_index(drop=True)

after tweet_id dedupe: 14481
after text dedupe: 14423


In [ ]:
print(df["airline_sentiment"].value_counts())
print(df["airline_clean"].value_counts())

airline_sentiment
negative    9079
neutral     3055
positive    2289
Name: count, dtype: int64
airline_clean
United            3805
US Airways        2906
American          2594
Southwest         2408
JetBlue           2207
Virgin America     503
Name: count, dtype: int64


In [ ]:
import re, html
def clean_text(t):
  t=html.unescape(t)
  t=re.sub(r"http\S+|ww\.\S+", "HTTPURL", t)
  t=re.sub(r"@\w+", "@USER", t)
  t=re.sub(r"\s+", " ", t).strip()
  return t

df["text_clean"]=df["text"].apply(clean_text)
df[["text", "text_clean"]].sample(5, random_state=1)

,text,text_clean
8119,@united We're hoping to hear from you actually.,@USER We're hoping to hear from you actually.
897,"@southwestair, Your male flight attendants mu...","@USER, Your male flight attendants must being ..."
8849,@AmericanAir My chariot tonight. From this ang...,@USER My chariot tonight. From this angle it l...
6122,"@AmericanAir Next Friday, I'll take AA3074 to ...","@USER Next Friday, I'll take AA3074 to LGA.Sat..."
10554,@USAirways I'm glad @Beamske retweeted otherwi...,@USER I'm glad @USER retweeted otherwise you w...


In [ ]:
key=(df["text_clean"].str.lower()
.str.replace(r"(@user|httpurl)", "", regex=True)
.str.replace(r"\s+", " ", regex=True).str.strip())

print("near duplicates to drop:", key.duplicated().sum())
df=df[~key.duplicated(keep="first")].reset_index(drop=True)
print("rows after:", len(df))
print(df["airline_sentiment"].value_counts())
print(df["airline_clean"].value_counts())

near duplicates to drop: 151
rows after: 14272
airline_sentiment
negative    9066
neutral     2979
positive    2227
Name: count, dtype: int64
airline_clean
United            3780
US Airways        2882
American          2574
Southwest         2384
JetBlue           2161
Virgin America     491
Name: count, dtype: int64


~ keeps rows from repeating. without keeps all the repeats.

In [ ]:
from sklearn.model_selection import train_test_split

SPEED=42
df["strat"]=df["airline_clean"]+"_"+ df["airline_sentiment"]

train_val, test=train_test_split(df, train_size=0.15, stratify=df["strat"], random_state=SPEED)
train, val=train_test_split(train_val, train_size=0.15/0.85, stratify=train_val["strat"], random_state=SPEED)

print(len(train), len(val), len(test))
for name, s in[("train", train), ("val", val), ("test", test)]:
  print(name, s["airline_sentiment"].value_counts(normalize=True).round(3).to_dict())


377 1763 12132
train {'negative': 0.634, 'neutral': 0.207, 'positive': 0.159}
val {'negative': 0.635, 'neutral': 0.209, 'positive': 0.155}
test {'negative': 0.635, 'neutral': 0.209, 'positive': 0.156}


In [ ]:
print(len(set(train.tweet_id)&set(test.tweet_id)),
      len(set(train.tweet_id)& set(val.tweet_id)),
      len(set(val.tweet_id)& set(test.tweet_id)))
print(len(set(train.text_clean) & set(test.text_clean)))

0 0 0
0


In [ ]:
import os
out="/content/drive/MyDrive/COMP1818"
os.makedirs(out, exist_ok=True)
train.to_csv(f"{out}/train.csv", index=False)
val.to_csv(f"{out}/val.csv", index=False)
test.to_csv(f"{out}/test.csv", index=False)
df.to_csv(f"{out}/cleaned_all.csv", index=False)

In [ ]:
print(os.listdir(out))

['multi_seed_val.json', 'tuning_val.json', 'cross_brand.json', 'test_predictions.csv', 'brand_perception.png', 'final_test.json', 'table_final_test.tex', 'table_cross_brand.tex', 'Tweets.csv', 'train.csv', 'val.csv', 'cleaned_all.csv', 'test.csv']


In [ ]:
#testing rule based polarity on microblog texts
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
LABELS = ["negative", "neutral", "positive"]
results=[]

def evaluate(y_true, y_pred, name):
  acc=accuracy_score(y_true, y_pred)
  mf1=f1_score(y_true, y_pred, average="macro", labels=LABELS)
  print(f"{name}: accuracy={acc:.3f} macro-F1={mf1:.3f}")
  print(classification_report(y_true, y_pred, labels=LABELS, digits=3, zero_division=0))
  print(confusion_matrix(y_true, y_pred, labels=LABELS))
  results.append({"model": name, "accuracy": acc, "macro_F1": mf1})

y_val=val["airline_sentiment"]

In [ ]:
majority=train["airline_sentiment"].mode()[0] #predicts most of the classses negatives for every instance
evaluate(y_val, [majority] * len(val), "majority class")

majority class: accuracy=0.635 macro-F1=0.259
              precision    recall  f1-score   support

    negative      0.635     1.000     0.777      1120
     neutral      0.000     0.000     0.000       369
    positive      0.000     0.000     0.000       274

    accuracy                          0.635      1763
   macro avg      0.212     0.333     0.259      1763
weighted avg      0.404     0.635     0.494      1763

[[1120    0    0]
 [ 369    0    0]
 [ 274    0    0]]


In [ ]:
rng=np.random.default_rng(42)
evaluate(y_val, rng.choice(LABELS, size=len(val)), "random (unifrom)")
props=train["airline_sentiment"].value_counts(normalize=True)
rng=np.random.default_rng(42)
evaluate(y_val, rng.choice(props.index.to_numpy(), size=len(val), p=props.to_numpy()), "random (class proportions)")

random (unifrom): accuracy=0.273 macro-F1=0.226
              precision    recall  f1-score   support

    negative      0.620     0.322     0.424      1120
     neutral      0.206     0.325     0.252       369
    postivie      0.000     0.000     0.000         0

   micro avg      0.273     0.323     0.296      1489
   macro avg      0.275     0.216     0.226      1489
weighted avg      0.518     0.323     0.382      1489

[[361 368 391]
 [138 120 111]
 [  0   0   0]]
random (class proportions): accuracy=0.470 macro-F1=0.273
              precision    recall  f1-score   support

    negative      0.638     0.630     0.634      1120
     neutral      0.185     0.182     0.183       369
    postivie      0.000     0.000     0.000         0

   micro avg      0.527     0.519     0.523      1489
   macro avg      0.274     0.271     0.273      1489
weighted avg      0.526     0.519     0.523      1489

[[706 234   0]
 [242  67   0]
 [  0   0   0]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia=SentimentIntensityAnalyzer() # calculatescontinuous compound sentiment polarity scores

for d in(train, val, test):
  d["vader"]=d["text_clean"].apply(lambda t: sia.polarity_scores(t)["compound"]) #converts scores into classes using a standard 0.05 threshold
def vader_label(c, t=0.05):
  return "postivie" if c >= t else ("negatvie" if c<= -t else "neutral")
pred =val["vader"].apply(vader_label)
evaluate(y_val, pred, "VADER (threshold 0.05)")

VADER (threshold 0.05): accuracy=0.082 macro-F1=0.129
              precision    recall  f1-score   support

    negative      0.000     0.000     0.000      1120
     neutral      0.386     0.390     0.388       369
    postivie      0.000     0.000     0.000         0

   micro avg      0.126     0.097     0.110      1489
   macro avg      0.129     0.130     0.129      1489
weighted avg      0.096     0.097     0.096      1489

[[  0 213 351]
 [  0 144 170]
 [  0   0   0]]


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
y_train=train["airline_sentiment"]

tfidf=TfidfVectorizer(ngram_range=(1,2), min_df=3, max_df=0.95, sublinear_tf=True, max_features=2000)
X_train=tfidf.fit_transform(train["text_clean"])
X_val=tfidf.transform(val["text_clean"])
print(X_train.shape)

lr=LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train, y_train)
evaluate(y_val, lr.predict(X_val), "TF-IDF + Linear Regression")

svm=LinearSVC(C=1.0, class_weight="balanced", random_state=42)
svm.fit(X_train, y_train)
evaluate(y_val, svm.predict(X_val), "TF-IDF + Linear SVM")

(377, 590)
TF-IDF + Linear Regression: accuracy=0.691 macro-F1=0.426
              precision    recall  f1-score   support

    negative      0.801     0.788     0.795      1120
     neutral      0.490     0.477     0.484       369
    postivie      0.000     0.000     0.000         0

   micro avg      0.725     0.711     0.718      1489
   macro avg      0.431     0.422     0.426      1489
weighted avg      0.724     0.711     0.718      1489

[[883 136   0]
 [151 176   0]
 [  0   0   0]]
TF-IDF + Linear SVM: accuracy=0.699 macro-F1=0.418
              precision    recall  f1-score   support

    negative      0.777     0.835     0.805      1120
     neutral      0.509     0.404     0.450       369
    postivie      0.000     0.000     0.000         0

   micro avg      0.724     0.728     0.726      1489
   macro avg      0.428     0.413     0.418      1489
weighted avg      0.710     0.728     0.717      1489

[[935 104   0]
 [184 149   0]
 [  0   0   0]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
for C in [0.1,0.3,1,3,10]:
  m= LogisticRegression(C=C, max_iter=2000, class_weight="balanced", random_state=42).fit(X_train, y_train)
  print("LR C =", C, round(f1_score(y_val, m.predict(X_val), average="macro"), 3))

for C in [0.1,0.3,1,3,10]:
  m= LinearSVC(C=C, class_weight="balanced", max_iter=5000, random_state=42).fit(X_train, y_train)
  print("SVM C =", C, round(f1_score(y_val, m.predict(X_val), average="macro"), 3))

LR C = 0.1 0.611
LR C = 0.3 0.61
LR C = 1 0.61
LR C = 3 0.606
LR C = 10 0.59
SVM C = 0.1 0.573
SVM C = 0.3 0.611
SVM C = 1 0.602
SVM C = 3 0.585
SVM C = 10 0.558


In [ ]:
import numpy as np
L=val["vader"].apply(vader_label).to_numpy()
M=lr.predict(X_val)

def union_neg(l, m):
  if l == "negative" or m=="negative":
    return "negative"
  if l == "positive" or m=="positive":
    return "positive"
  return "neutral"

def union_pos(l, m):
   if l == "positive" or m=="positive":
    return "positive"
   if l == "negative" or m=="negative": return "negative"
   return "neutral"

def agree_else_neutral(l, m):
  return l if l == m else "neutral"

evaluate(y_val, np.array([union_neg(l,m) for l,m in zip(L,M)]), "Union (negative priority)")
evaluate(y_val, np.array([union_pos(l,m) for l,m in zip(L,M)]), "Union (positive priority)")
evaluate(y_val, np.array([agree_else_neutral(l,m) for l,m in zip(L,M)]), "Agree else neutral")
print("Lexicon and ML agree on:", round((L==M).mean(),3))

Union (negative priority): accuracy=0.691 macro-F1=0.426
              precision    recall  f1-score   support

    negative      0.801     0.788     0.795      1120
     neutral      0.490     0.477     0.484       369
    postivie      0.000     0.000     0.000         0

   micro avg      0.725     0.711     0.718      1489
   macro avg      0.431     0.422     0.426      1489
weighted avg      0.724     0.711     0.718      1489

[[883 136   0]
 [151 176   0]
 [  0   0   0]]
Union (positive priority): accuracy=0.691 macro-F1=0.426
              precision    recall  f1-score   support

    negative      0.801     0.788     0.795      1120
     neutral      0.490     0.477     0.484       369
    postivie      0.000     0.000     0.000         0

   micro avg      0.725     0.711     0.718      1489
   macro avg      0.431     0.422     0.426      1489
weighted avg      0.724     0.711     0.718      1489

[[883 136   0]
 [151 176   0]
 [  0   0   0]]
Agree else neutral: accuracy=0.2

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import random, time, numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

MODEL_NAME="roberta-base"
MAX_LEN=80
label2id={"negative": 0, "neutral": 1, "positive": 2}
id2label={v: k for k, v in label2id.items()}
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)

lens=[len(x) for x in tokenizer(train["text_clean"].tolist())["input_ids"]]
print(np.percentile(lens, [50,95,99,100]))

device: cuda


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[27. 39. 44. 48.]


In [ ]:
def set_seed(seed):
  random.seed(seed); np.random.seed(seed)
  torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def encode(d, extra=None):
  enc = tokenizer(d["text_clean"].tolist(), truncation=True, padding="max_length",
                  max_length=MAX_LEN, return_tensors="pt")
  labels = torch.tensor(d["airline_sentiment"].map(label2id).values)
  if extra is None:
    extra = torch.zeros(len(d), 0)
  return TensorDataset(enc["input_ids"], enc["attention_mask"], extra, labels)

class SentimentModels(nn.Module):
  def __init__(self, name, n_extra=0, n_classes=3, dropout=0.1):
    super().__init__()
    self.encoder=AutoModel.from_pretrained(name)
    self.drop=nn.Dropout(dropout)
    self.classifier=nn.Linear(self.encoder.config.hidden_size + n_extra, n_classes)
  def forward(self, input_ids, attention_mask, extra):
    h=self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:,0]
    return self.classifier(self.drop(torch.cat([h, extra.to(h.dtype)], dim=1)))

def trim(ids, mask):
  n=int(mask.sum(1).max())
  return ids[:, :n], mask[:, :n]

@torch.no_grad()
def predict(model, loader):
  model.eval(); preds, trues =[], []
  for ids, mask, extra, y in loader:
    ids, mask=trim(ids, mask)
    with torch.autocast("cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
      logits=model(ids.to(device), mask.to(device), extra.to(device).float())
    preds.append(logits.argmax(dim=1).cpu()); trues.append(y)
  return torch.cat(preds).numpy(), torch.cat(trues).numpy()

def train_model(train_ds, val_ds, epochs=3, lr=2e-5, batch_size=32, seed=42, n_extra=0, head_lr=None):
  set_seed(seed)
  use_amp=torch.cuda.is_available()
  model=SentimentModels(MODEL_NAME, n_extra=n_extra).to(device)
  g=torch.Generator(); g.manual_seed(seed)
  tr=DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=g)
  va=DataLoader(val_ds, batch_size=128)
  opt=torch.optim.AdamW([{"params":model.encoder.parameters(), "lr":lr},
                        {"params":model.classifier.parameters(), "lr":head_lr},
                        ], weight_decay=0.01)
  total=len(tr) * epochs
  sched=get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
  scaler=torch.amp.GradScaler("cuda", enabled=use_amp)

  best_f1, best_state, best_pred=-1, None, None
  t0=time.time()
  for ep in range(1, epochs+1):
    model.train()
    for step, (ids, mask, extra, y) in enumerate(tr, 1):
      ids, mask=trim(ids, mask)
      ids, mask, extra, y=ids.to(device), mask.to(device), extra.to(device).float(), y.to(device)
      opt.zero_grad()
      with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
        logits=model(ids, mask, extra)
        loss=nn.functional.cross_entropy(logits.float(), y)
      scaler.scale(loss).backward()
      scaler.unscale_(opt)
      nn.utils.clip_grad_norm_(model.parameters(), 1.0)
      scaler.step(opt); scaler.update(); sched.step()
      if step % 50 == 0 or step == len(tr):
        print(f" epoch {ep} step {step}/{len(tr)} loss {loss.item():.3f} {time.time()-t0:.0f}s")

    pred, true=predict(model, va)
    f1=f1_score(true, pred, average="macro")
    print(f" epoch {ep}: val macro-F1 = {f1:.3f}")
    if f1 > best_f1:
      best_f1, best_pred = f1, pred
      best_state={k:v.detach().cpu().clone() for k, v in model.state_dict().items()}

  model.load_state_dict(best_state)
  return model, best_pred

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia=SentimentIntensityAnalyzer()
def vader_feats(d):
  s=d["text_clean"].apply(sia.polarity_scores)
  return torch.tensor([[x["neg"], x["neu"], x["pos"], x["compound"]] for x in s],)

Xv_tr, Xv_val=vader_feats(train), vader_feats(val)
train_h, val_h=encode(train, Xv_tr), encode(val, Xv_val)
print(train_h.tensors[2].shape, val_h.tensors[2].shape)

torch.Size([377, 4]) torch.Size([1763, 4])


In [ ]:
m_t2, p_t2 = train_model(encode(train), encode(val), epochs=3, lr=2e-5, seed=42, n_extra=0, head_lr=1e-3)
evaluate(y_val, [id2label[i] for i in p_t2], "RoBERTa only (head lr 1e-3)")

m_h2, p_h2 = train_model(train_h, val_h, epochs=3, lr=2e-5, seed=42, n_extra=4, head_lr=1e-3)
evaluate(y_val, [id2label[i] for i in p_h2], "RoBERTa + Vader (head lr 1e-3)")

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_981/2360883953.py:65: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the lea

 epoch 1 step 12/12 loss 1.058 3s
 epoch 1: val macro-F1 = 0.259
 epoch 2 step 12/12 loss 0.766 6s
 epoch 2: val macro-F1 = 0.259
 epoch 3 step 12/12 loss 0.915 8s
 epoch 3: val macro-F1 = 0.259
RoBERTa only (head lr 1e-3): accuracy=0.635 macro-F1=0.259
              precision    recall  f1-score   support

    negative      0.635     1.000     0.777      1120
     neutral      0.000     0.000     0.000       369
    postivie      0.000     0.000     0.000         0

   micro avg      0.635     0.752     0.689      1489
   macro avg      0.212     0.333     0.259      1489
weighted avg      0.478     0.752     0.584      1489



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[[1120    0    0]
 [ 369    0    0]
 [   0    0    0]]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_981/2360883953.py:65: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the lea

 epoch 1 step 12/12 loss 1.073 1s
 epoch 1: val macro-F1 = 0.259
 epoch 2 step 12/12 loss 0.764 4s
 epoch 2: val macro-F1 = 0.259
 epoch 3 step 12/12 loss 0.821 6s
 epoch 3: val macro-F1 = 0.259
RoBERTa + Vader (head lr 1e-3): accuracy=0.635 macro-F1=0.259
              precision    recall  f1-score   support

    negative      0.635     1.000     0.777      1120
     neutral      0.000     0.000     0.000       369
    postivie      0.000     0.000     0.000         0

   micro avg      0.635     0.752     0.689      1489
   macro avg      0.212     0.333     0.259      1489
weighted avg      0.478     0.752     0.584      1489

[[1120    0    0]
 [ 369    0    0]
 [   0    0    0]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import json
from IPython.display import clear_output

y_val_ids = val["airline_sentiment"].map(label2id).values
tr_t, va_t = encode(train), encode(val)
configs = {"transformer": (tr_t, va_t, 0), "hybrid": (train_h, val_h, 4)}
multi = {k: [] for k in configs}

for seed in [1, 2, 3, 4, 5]:
    for name, (trd, vad, ne) in configs.items():
        _, p = train_model(trd, vad, epochs=3, lr=2e-5, seed=seed, n_extra=ne, head_lr=1e-3)
        multi[name].append(float(f1_score(y_val_ids, p, average="macro")))
        json.dump(multi, open("/content/drive/MyDrive/COMP1818/multi_seed_val.json", "w"))
    clear_output(wait=True)
    print(f"finished seed {seed}")
    for k, v in multi.items():
        print(k, [round(x, 4) for x in v])

for k, v in multi.items():
    print(f"{k}: mean {np.mean(v):.4f}  std {np.std(v, ddof=1):.4f}")
d = np.array(multi["hybrid"]) - np.array(multi["transformer"])
print(f"paired difference (hybrid - transformer): mean {d.mean():+.4f}  std {d.std(ddof=1):.4f}")

finished seed 5
transformer [0.259, 0.259, 0.259, 0.259, 0.259]
hybrid [0.259, 0.3623, 0.259, 0.259, 0.259]
transformer: mean 0.2590  std 0.0000
hybrid: mean 0.2797  std 0.0462
paired difference (hybrid - transformer): mean +0.0207  std 0.0462


In [ ]:
tune = {}
for lr_ in [1e-5, 3e-5]:
    tune[lr_] = []
    for seed in [1, 2, 3]:
        _, p = train_model(tr_t, va_t, epochs=3, lr=lr_, seed=seed, n_extra=0, head_lr=1e-3)
        tune[lr_].append(float(f1_score(y_val_ids, p, average="macro")))
    json.dump({str(k): v for k, v in tune.items()},
              open("/content/drive/MyDrive/COMP1818/tuning_val.json", "w"))
clear_output(wait=True)
base3 = multi["transformer"][:3]
print(f"lr 2e-5: {[round(x,4) for x in base3]}  mean {np.mean(base3):.4f}")
for k, v in tune.items():
    print(f"lr {k:g}: {[round(x,4) for x in v]}  mean {np.mean(v):.4f}")

lr 2e-5: [0.259, 0.259, 0.259]  mean 0.2590
lr 1e-05: [0.259, 0.259, 0.259]  mean 0.2590
lr 3e-05: [0.259, 0.5985, 0.259]  mean 0.3722


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

LABELS = ["negative", "neutral", "positive"]
all_df = pd.read_csv("/content/drive/MyDrive/COMP1818/cleaned_all.csv")
Xall = vader_feats(all_df)
all_df["vader"] = Xall[:, 3].numpy()
brands = sorted(all_df["airline_clean"].unique())
print(brands, len(all_df))

def vl(c): return "positive" if c >= 0.05 else ("negative" if c <= -0.05 else "neutral")
def mf1(y, p): return f1_score(y, p, average="macro", labels=LABELS, zero_division=0)

base = {}
for b in brands:
    tr_b = train[train["airline_clean"] != b]
    ho = all_df[all_df["airline_clean"] == b]
    y = ho["airline_sentiment"]
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.95, sublinear_tf=True, max_features=20000)
    clf = LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced", random_state=42)
    clf.fit(vec.fit_transform(tr_b["text_clean"]), tr_b["airline_sentiment"])
    base[b] = {"n": len(ho),
               "majority": mf1(y, ["negative"] * len(y)),
               "vader": mf1(y, ho["vader"].apply(vl)),
               "tfidf_lr": mf1(y, clf.predict(vec.transform(ho["text_clean"])))}
print(pd.DataFrame(base).T.round(3))

['American', 'JetBlue', 'Southwest', 'US Airways', 'United', 'Virgin America'] 14272
                     n  majority  vader  tfidf_lr
American        2574.0     0.280  0.474     0.565
JetBlue         2161.0     0.204  0.568     0.601
Southwest       2384.0     0.221  0.548     0.594
US Airways      2882.0     0.293  0.429     0.551
United          3780.0     0.273  0.487     0.559
Virgin America   491.0     0.179  0.602     0.583


In [ ]:
cross = {b: {"transformer": [], "hybrid": []} for b in brands}

def show():
    rows = []
    for b in brands:
        t, h = cross[b]["transformer"], cross[b]["hybrid"]
        if t and h:
            rows.append({"brand": b, **base[b], "roberta": np.mean(t), "hybrid": np.mean(h),
                         "hyb-rob": np.mean(h) - np.mean(t)})
    out = pd.DataFrame(rows).set_index("brand")
    print(out.round(3)); print("mean over brands:\n", out.drop(columns="n").mean().round(3))

for b in brands:
    m_tr = (train["airline_clean"] != b).values
    m_va = (val["airline_clean"] != b).values
    tr_b, va_b = train[m_tr], val[m_va]
    ho = all_df[all_df["airline_clean"] == b]
    ho_x = Xall[torch.tensor(ho.index.values)]
    sets = {"transformer": (encode(tr_b), encode(va_b), encode(ho), 0),
            "hybrid": (encode(tr_b, Xv_tr[torch.tensor(m_tr)]),
                       encode(va_b, Xv_val[torch.tensor(m_va)]),
                       encode(ho, ho_x), 4)}
    for seed in [1, 2]:
        for name, (trd, vad, hod, ne) in sets.items():
            model, _ = train_model(trd, vad, epochs=3, lr=2e-5, seed=seed, n_extra=ne, head_lr=1e-3)
            pred, true = predict(model, DataLoader(hod, batch_size=128))
            cross[b][name].append(float(f1_score(true, pred, average="macro")))
            del model; torch.cuda.empty_cache()
    json.dump({"cross": cross, "base": base},
              open("/content/drive/MyDrive/COMP1818/cross_brand.json", "w"))
    clear_output(wait=True)
    print(f"done: {b}"); show()

done: Virgin America
                   n  majority  vader  tfidf_lr  roberta  hybrid  hyb-rob
brand                                                                    
American        2574     0.280  0.474     0.565    0.280   0.280      0.0
JetBlue         2161     0.204  0.568     0.601    0.204   0.204      0.0
Southwest       2384     0.221  0.548     0.594    0.221   0.221      0.0
US Airways      2882     0.293  0.429     0.551    0.293   0.293      0.0
United          3780     0.273  0.487     0.559    0.273   0.273      0.0
Virgin America   491     0.179  0.602     0.583    0.179   0.179      0.0
mean over brands:
 majority    0.242
vader       0.518
tfidf_lr    0.575
roberta     0.242
hybrid      0.242
hyb-rob     0.000
dtype: float64


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

y_te = test["airline_sentiment"].to_numpy()
Xv_te = vader_feats(test)
test_ds, test_ds_h = encode(test), encode(test, Xv_te)
final, preds = {}, {}

def rec(name, y, p):
    final.setdefault(name, []).append((accuracy_score(y, p), mf1(y, p)))

def summary():
    rows = []
    for name, v in final.items():
        a = np.array(v)
        rows.append({"model": name, "runs": len(v), "accuracy": a[:, 0].mean(),
                     "macro_F1": a[:, 1].mean(),
                     "std": a[:, 1].std(ddof=1) if len(v) > 1 else 0.0})
    print(pd.DataFrame(rows).round(4).to_string(index=False))

# trivial + classical baselines
rec("Majority class", y_te, ["negative"] * len(y_te))
rec("Random (uniform)", y_te, np.random.default_rng(42).choice(LABELS, len(y_te)))
rec("VADER only", y_te, [vl(c) for c in Xv_te[:, 3].numpy()])
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.95, sublinear_tf=True, max_features=20000)
Xtr_t, Xte_t = vec.fit_transform(train["text_clean"]), vec.transform(test["text_clean"])
ytr = train["airline_sentiment"]
rec("TF-IDF + LogReg", y_te, LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced", random_state=42).fit(Xtr_t, ytr).predict(Xte_t))
rec("TF-IDF + LinearSVC", y_te, LinearSVC(C=1.0, class_weight="balanced", random_state=42).fit(Xtr_t, ytr).predict(Xte_t))

# transformer models, 5 seeds
jobs = [("RoBERTa", tr_t, va_t, test_ds, 0), ("RoBERTa + VADER", train_h, val_h, test_ds_h, 4)]
for seed in [1, 2, 3, 4, 5]:
    for name, trd, vad, ted, ne in jobs:
        model, _ = train_model(trd, vad, epochs=3, lr=2e-5, seed=seed, n_extra=ne, head_lr=1e-3)
        p, _ = predict(model, DataLoader(ted, batch_size=128))
        p = [id2label[i] for i in p]
        rec(name, y_te, p); preds[f"{name}_seed{seed}"] = p
        del model; torch.cuda.empty_cache()
    out = test[["tweet_id", "airline_clean", "airline_sentiment", "text_clean"]].copy()
    for k, v in preds.items(): out[k] = v
    out.to_csv("/content/drive/MyDrive/COMP1818/test_predictions.csv", index=False)
    json.dump(final, open("/content/drive/MyDrive/COMP1818/final_test.json", "w"))
    clear_output(wait=True); print(f"finished seed {seed}"); summary()

finished seed 5
             model  runs  accuracy  macro_F1   std
    Majority class     1    0.6353    0.2590 0.000
  Random (uniform)     1    0.3306    0.2981 0.000
        VADER only     1    0.5500    0.5117 0.000
   TF-IDF + LogReg     1    0.6819    0.5960 0.000
TF-IDF + LinearSVC     1    0.6888    0.5837 0.000
           RoBERTa     5    0.6353    0.2590 0.000
   RoBERTa + VADER     5    0.6406    0.2796 0.046


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

pr = pd.read_csv("/content/drive/MyDrive/COMP1818/test_predictions.csv")
pr["conf"] = test["airline_sentiment_confidence"].values
pr["words"] = test["text_clean"].str.split().str.len().values
pr["len_bucket"] = pd.cut(pr["words"], [0, 8, 15, 200], labels=["short (<=8)", "medium (9-15)", "long (16+)"])
pr["conf_bucket"] = pd.cut(pr["conf"], [0, 0.7, 0.99, 1.0], labels=["<=0.70", "0.70-0.99", "1.0"])
pr["negation"] = pr["text_clean"].str.contains(r"\b(?:not|no|never|cannot)\b|n't", case=False, regex=True)
pr["question"] = pr["text_clean"].str.contains(r"\?")
cols = {n: [f"{n}_seed{s}" for s in range(1, 6)] for n in ["RoBERTa", "RoBERTa + VADER"]}

#per-airline macro-F1 (mean over 5 seeds)
rows = []
for b, g in pr.groupby("airline_clean"):
    r = {"airline": b, "n": len(g)}
    for n, cs in cols.items():
        r[n] = np.mean([mf1(g["airline_sentiment"], g[c]) for c in cs])
    rows.append(r)
print(pd.DataFrame(rows).round(3).to_string(index=False))

# brand perception: gold vs predicted sentiment share (RoBERTa, seed 1)
gold = pd.crosstab(pr["airline_clean"], pr["airline_sentiment"], normalize="index")[LABELS]
pred = pd.crosstab(pr["airline_clean"], pr["RoBERTa_seed1"], normalize="index")[LABELS]
print((gold * 100).round(1)); print((pred * 100).round(1))
fig, ax = plt.subplots(1, 3, figsize=(12, 3.3), sharey=True)
for a, lab in zip(ax, LABELS):
    x = np.arange(len(gold))
    a.bar(x - 0.2, gold[lab], 0.4, label="human label"); a.bar(x + 0.2, pred[lab], 0.4, label="RoBERTa")
    a.set_xticks(x); a.set_xticklabels(gold.index, rotation=45, ha="right"); a.set_title(lab)
ax[0].legend(); plt.tight_layout()
plt.savefig("/content/drive/MyDrive/COMP1818/brand_perception.png", dpi=200); plt.show()

# error slices: accuracy averaged over 5 seeds
for col in ["len_bucket", "conf_bucket", "negation", "question"]:
    print(f"\n--- accuracy by {col}")
    for k, g in pr.groupby(col, observed=True):
        acc = np.mean([(g[c] == g["airline_sentiment"]).mean() for c in cols["RoBERTa"]])
        print(f"{str(k):16s} n={len(g):5d}  accuracy={acc:.3f}")

# class report, confusion matrix and example errors (RoBERTa, seed 1)
print(classification_report(pr["airline_sentiment"], pr["RoBERTa_seed1"], digits=3))
print(confusion_matrix(pr["airline_sentiment"], pr["RoBERTa_seed1"], labels=LABELS))
pd.set_option("display.max_colwidth", 140)
err = pr[pr["RoBERTa_seed1"] != pr["airline_sentiment"]]
for (g_, p_), grp in err.groupby(["airline_sentiment", "RoBERTa_seed1"]):
    print(f"\n= human: {g_}  model: {p_}  ({len(grp)} errors)")
    print(grp.sample(min(5, len(grp)), random_state=0)[["text_clean", "conf"]].to_string(index=False))

       airline    n  RoBERTa  RoBERTa + VADER
      American 2188    0.280            0.304
       JetBlue 1838    0.204            0.227
     Southwest 2026    0.221            0.241
    US Airways 2449    0.293            0.313
        United 3214    0.273            0.291
Virgin America  417    0.179            0.199


KeyError: "['neutral', 'positive'] not in index"

In [ ]:
out_dir = "/content/drive/MyDrive/COMP1818/"
def tex(df, name):
    s = df.to_latex(index=False, float_format="%.3f")
    open(out_dir + name, "w").write(s); print(s)

rows = []
for name, v in final.items():
    a = np.array(v)
    rows.append({"model": name, "accuracy": a[:, 0].mean(), "macro_F1": a[:, 1].mean(),
                 "std": a[:, 1].std(ddof=1) if len(v) > 1 else 0.0})
tex(pd.DataFrame(rows), "table_final_test.tex")

cb = pd.DataFrame([{"airline": b, "n": base[b]["n"], "majority": base[b]["majority"],
                    "vader": base[b]["vader"], "tfidf_lr": base[b]["tfidf_lr"],
                    "roberta": np.mean(cross[b]["transformer"]),
                    "hybrid": np.mean(cross[b]["hybrid"])} for b in brands])
tex(cb, "table_cross_brand.tex")

\begin{tabular}{lrrr}
\toprule
model & accuracy & macro_F1 & std \\
\midrule
Majority class & 0.635 & 0.259 & 0.000 \\
Random (uniform) & 0.331 & 0.298 & 0.000 \\
VADER only & 0.550 & 0.512 & 0.000 \\
TF-IDF + LogReg & 0.682 & 0.596 & 0.000 \\
TF-IDF + LinearSVC & 0.689 & 0.584 & 0.000 \\
RoBERTa & 0.635 & 0.259 & 0.000 \\
RoBERTa + VADER & 0.641 & 0.280 & 0.046 \\
\bottomrule
\end{tabular}

\begin{tabular}{lrrrrrr}
\toprule
airline & n & majority & vader & tfidf_lr & roberta & hybrid \\
\midrule
American & 2574 & 0.280 & 0.474 & 0.565 & 0.280 & 0.280 \\
JetBlue & 2161 & 0.204 & 0.568 & 0.601 & 0.204 & 0.204 \\
Southwest & 2384 & 0.221 & 0.548 & 0.594 & 0.221 & 0.221 \\
US Airways & 2882 & 0.293 & 0.429 & 0.551 & 0.293 & 0.293 \\
United & 3780 & 0.273 & 0.487 & 0.559 & 0.273 & 0.273 \\
Virgin America & 491 & 0.179 & 0.602 & 0.583 & 0.179 & 0.179 \\
\bottomrule
\end{tabular}



In [ ]:
from sklearn.metrics import accuracy_score
import json


lr_final = LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced", random_state=42).fit(Xtr_t, ytr)
M_val = lr_final.predict(vec.transform(val["text_clean"]))
M_te  = lr_final.predict(Xte_t)
L_val = [vl(c) for c in Xv_val[:, 3].numpy()]
L_te  = [vl(c) for c in Xv_te[:, 3].numpy()]
y_va  = val["airline_sentiment"]

def combine(L, M, rule):
    out = []
    for l, m in zip(L, M):
        if rule == "agree_else_neutral":
            out.append(l if l == m else "neutral")
        elif rule == "union_neg":
            out.append("negative" if "negative" in (l, m) else ("positive" if "positive" in (l, m) else "neutral"))
        else:
            out.append("positive" if "positive" in (l, m) else ("negative" if "negative" in (l, m) else "neutral"))
    return out

for rule in ["agree_else_neutral", "union_neg", "union_pos"]:
    pv, pt = combine(L_val, M_val, rule), combine(L_te, M_te, rule)
    print(f"{rule}: val macro-F1 {mf1(y_va, pv):.3f} | test macro-F1 {mf1(y_te, pt):.3f}")
    final[f"Union ({rule.replace('_', ' ')})"] = [(accuracy_score(y_te, pt), mf1(y_te, pt))]
json.dump(final, open(out_dir + "final_test.json", "w"))

def tex(df, name):
    s = df.to_latex(index=False, float_format="%.3f", na_rep="-", escape=True)
    open(out_dir + name, "w").write(s); print(s)

rows = []
for name, v in final.items():
    a = np.array(v)
    rows.append({"Model": name, "Accuracy": a[:, 0].mean(), "Macro-F1": a[:, 1].mean(),
                 "Std": a[:, 1].std(ddof=1) if len(v) > 1 else np.nan})
tex(pd.DataFrame(rows), "table_final_test.tex")

cb = pd.DataFrame([{"Airline": b, "N": str(base[b]["n"]), "Majority": base[b]["majority"],
                    "VADER": base[b]["vader"], "TF-IDF LR": base[b]["tfidf_lr"],
                    "RoBERTa": np.mean(cross[b]["transformer"]),
                    "Hybrid": np.mean(cross[b]["hybrid"])} for b in brands])
mean_row = {"Airline": "Mean", "N": "",
            **{c: cb[c].mean() for c in ["Majority", "VADER", "TF-IDF LR", "RoBERTa", "Hybrid"]}}
cb = pd.concat([cb, pd.DataFrame([mean_row])], ignore_index=True)
tex(cb, "table_cross_brand.tex")

agree_else_neutral: val macro-F1 0.540 | test macro-F1 0.541
union_neg: val macro-F1 0.568 | test macro-F1 0.566
union_pos: val macro-F1 0.502 | test macro-F1 0.500
\begin{tabular}{lrrr}
\toprule
Model & Accuracy & Macro-F1 & Std \\
\midrule
Majority class & 0.635 & 0.259 & - \\
Random (uniform) & 0.331 & 0.298 & - \\
VADER only & 0.550 & 0.512 & - \\
TF-IDF + LogReg & 0.682 & 0.596 & - \\
TF-IDF + LinearSVC & 0.689 & 0.584 & - \\
RoBERTa & 0.635 & 0.259 & 0.000 \\
RoBERTa + VADER & 0.641 & 0.280 & 0.046 \\
Union (agree else neutral) & 0.529 & 0.541 & - \\
Union (union neg) & 0.705 & 0.566 & - \\
Union (union pos) & 0.585 & 0.500 & - \\
Union ( a g r e e \_ e l s e \_ n e u t r a l ) & 0.529 & 0.541 & - \\
Union ( u n i o n \_ n e g ) & 0.705 & 0.566 & - \\
Union ( u n i o n \_ p o s ) & 0.585 & 0.500 & - \\
\bottomrule
\end{tabular}

\begin{tabular}{llrrrrr}
\toprule
Airline & N & Majority & VADER & TF-IDF LR & RoBERTa & Hybrid \\
\midrule
American & 2574 & 0.280 & 0.474 & 0.565 & 0.2